In [7]:
import numpy as np

from MAM import MAM
from MAMH import MAMH
from MAMR import MAMR

# Para obtener siempre los mismos datos aleatorios
rng = np.random.default_rng(7)

# Número de medidas
M = 2

# Dimensión de p. Se escoge R=4 porque 4=2² y el programa genera
# imágenes de tamaño 2x2.
R = 4

# Longitud de cada vector q[m]
S = np.array([3, 5])

# Vector de probabilidad inicial
p = np.array([0.1, 0.2, 0.3, 0.4])

# Construcción de los vectores q
q = []

for s in S:
    vector = rng.random(s)
    vector = vector / vector.sum()  # La suma queda igual a 1
    q.append(vector)

# Construcción de las matrices de costo d
d = []

for s in S:
    matriz_costo = 0.05 * rng.random((R, s))
    d.append(matriz_costo)

rho = 1.0

In [8]:
print("p:", p.shape)

for m in range(M):
    print(f"q[{m}]:", q[m].shape)
    print(f"d[{m}]:", d[m].shape)

p: (4,)
q[0]: (3,)
d[0]: (4, 3)
q[1]: (5,)
d[1]: (4, 5)


In [9]:
algoritmos = {
    "MAM": MAM,
    "MAMH": MAMH,
    "MAMR": MAMR
}

for nombre, algoritmo in algoritmos.items():

    print(f"\nProbando {nombre}")

    p_resultado, val, cpu, theta = algoritmo(
        d=d,
        q=q,
        M=M,
        R=R,
        S=S,
        p=p,
        rho=rho,
        UseGPU=False,
        tol=-1.0,
        MaxCPU=0.0,
        PrintEvery=1.0,
        save_figures=False
    )

    print("p obtenido:", p_resultado)
    print("Suma de p:", np.sum(p_resultado))
    print("val:", val)
    print("Tiempo:", cpu)
    print("Dimensiones de theta:", [matriz.shape for matriz in theta])

    # Comprobaciones automáticas
    assert p_resultado.shape == (R,)
    assert np.all(np.isfinite(p_resultado))
    assert all(np.all(np.isfinite(matriz)) for matriz in theta)
    assert theta[0].shape == (R, S[0])
    assert theta[1].shape == (R, S[1])

    print(f"{nombre} pasó la prueba básica.")


Probando MAM
k =     1, |pk-pkk| = 0.00e+00, cpu =     0
k =     1, |pk-pkk| = 0.00e+00, cpu =     0
p obtenido: [0.1 0.2 0.3 0.4]
Suma de p: 1.0
val: 0.0
Tiempo: 0.0009951000101864338
Dimensiones de theta: [(4, 3), (4, 5)]
MAM pasó la prueba básica.

Probando MAMH
k =     1, |pk-pkk| = 0.00e+00, cpu =     0
k =     1, |pk-pkk| = 0.00e+00, cpu =     0
p obtenido: [0.1 0.2 0.3 0.4]
Suma de p: 1.0
val: 0.0
Tiempo: 0.00034900009632110596
Dimensiones de theta: [(4, 3), (4, 5)]
MAMH pasó la prueba básica.

Probando MAMR
k =     1, |pk-pkk| = 0.00e+00, cpu =     0
k =     1, |pk-pkk| = 0.00e+00, cpu =     0
p obtenido: [0.1 0.2 0.3 0.4]
Suma de p: 1.0
val: 0.0
Tiempo: 0.00027219997718930244
Dimensiones de theta: [(4, 3), (4, 5)]
MAMR pasó la prueba básica.


In [10]:
save_figures=True
output_dir="Figuras"

In [11]:
carpeta_figuras = (
    r"C:\Users\ignac\Downloads\Matlab - MAMR-20260906T134616Z-1-001"
    r"\Matlab - MAMR\Fig"
)

p_resultado, val, cpu, theta = MAMR(
    d=d,
    q=q,
    M=M,
    R=R,
    S=S,
    p=p,
    rho=rho,
    UseGPU=False,
    tol=-1.0,
    MaxCPU=2.0,
    PrintEvery=0.5,
    save_figures=True,
    output_dir=carpeta_figuras
)

k =     1, |pk-pkk| = 0.00e+00, cpu =     0
k =  5854, |pk-pkk| = 6.94e-18, cpu =     1
k = 11145, |pk-pkk| = 6.94e-18, cpu =     1
k = 16496, |pk-pkk| = 0.00e+00, cpu =     2
k = 22046, |pk-pkk| = 0.00e+00, cpu =     2
k = 22046, |pk-pkk| = 0.00e+00, cpu =     2


In [13]:
"""Conversión a Python de ``distGrid.m``."""

import numpy as np
from scipy.spatial.distance import cdist


def distGrid(K, M):
    """Calcula las distancias euclidianas entre dos grillas.

    La primera grilla tiene paso ``1/M`` y la segunda tiene paso uno. El
    resultado posee dimensiones

    ``((M * (K - 1) + 1)**2, K**2)``.

    Esta implementación conserva el orden por columnas de MATLAB.
    """
    K = int(K)
    M = int(M)
    if K < 1:
        raise ValueError("K debe ser un entero positivo.")
    if M < 1:
        raise ValueError("M debe ser un entero positivo.")

    fine_size = M * (K - 1) + 1
    fine_axis = np.linspace(1.0, float(K), fine_size)
    base_axis = np.arange(1.0, K + 1.0)

    fine_x, fine_y = np.meshgrid(fine_axis, fine_axis, indexing="xy")
    base_x, base_y = np.meshgrid(base_axis, base_axis, indexing="xy")

    fine_points = np.column_stack(
        (fine_x.ravel(order="F"), fine_y.ravel(order="F"))
    )
    base_points = np.column_stack(
        (base_x.ravel(order="F"), base_y.ravel(order="F"))
    )

    return cdist(fine_points, base_points, metric="euclidean")


dist_grid = distGrid

__all__ = ["distGrid", "dist_grid"]

In [14]:
"""Conversión a Python de ``LP_WB.m``."""

from time import perf_counter

import numpy as np
from scipy import sparse
from scipy.optimize import linprog


def LP_WB(D, Q, M, R, S, display=True, method="highs"):
    """Resuelve el baricentro de Wasserstein como un programa lineal.

    El original llama a ``linprogGurobi``. Esta versión usa
    ``scipy.optimize.linprog`` con HiGHS, por lo que no requiere Gurobi.

    Retorna ``(p, val, cpu, m, n)`` como la función MATLAB.
    """
    start = perf_counter()
    M = int(M)
    R = int(R)
    S = np.asarray(S, dtype=int).reshape(-1)

    if len(D) != M or len(Q) != M or S.size != M:
        raise ValueError("M debe coincidir con las longitudes de D, Q y S.")

    costs = []
    constraint_blocks = []
    barycenter_blocks = []
    rhs = []

    for index in range(M):
        s = int(S[index])
        distance = np.asarray(D[index], dtype=float)
        q = np.asarray(Q[index], dtype=float).reshape(-1)

        if distance.shape != (R, s):
            raise ValueError(
                f"D[{index}] debe tener forma {(R, s)}; "
                f"tiene {distance.shape}."
            )
        if q.size != s:
            raise ValueError(f"Q[{index}] debe tener longitud {s}.")

        costs.append(distance.reshape(R * s, order="F"))

        column_sums = sparse.kron(
            sparse.eye(s, format="csr"), np.ones((1, R)), format="csr"
        )
        row_sums = sparse.kron(
            np.ones((1, s)), sparse.eye(R, format="csr"), format="csr"
        )
        constraint_blocks.append(
            sparse.vstack((column_sums, row_sums), format="csr")
        )
        barycenter_blocks.append(
            sparse.vstack(
                (sparse.csr_matrix((s, R)), -sparse.eye(R, format="csr")),
                format="csr",
            )
        )
        rhs.extend((q, np.zeros(R, dtype=float)))

    objective = np.concatenate((*costs, np.zeros(R, dtype=float)))
    transport_constraints = sparse.block_diag(
        constraint_blocks, format="csr"
    )
    barycenter_constraints = sparse.vstack(
        barycenter_blocks, format="csr"
    )
    A_eq = sparse.hstack(
        (transport_constraints, barycenter_constraints), format="csr"
    )
    b_eq = np.concatenate(rhs)

    rows, columns = A_eq.shape
    result = linprog(
        objective,
        A_eq=A_eq,
        b_eq=b_eq,
        bounds=(0.0, None),
        method=method,
        options={"disp": bool(display)},
    )

    if not result.success:
        raise RuntimeError(
            f"El programa lineal no pudo resolverse: {result.message}"
        )

    p = result.x[-R:]
    cpu = perf_counter() - start
    return p, float(result.fun), cpu, rows, columns


lp_wb = LP_WB

__all__ = ["LP_WB", "lp_wb"]


In [17]:
"""Conversión a Python de ``main.m``.

Este es el punto de entrada del proyecto. Debe estar en la misma carpeta que
los demás módulos convertidos y que las carpetas ``dataPeyre`` o
``dataAltschuler``.
"""

from pathlib import Path

import numpy as np

from GetPlan import GetPlan
from LP_WB import LP_WB
from MAM import MAM
from MAMH import MAMH
from MAMR import MAMR
from bregmanWassersteinBarycenter import bregmanWassersteinBarycenter
from distGrid import distGrid
from salvaPNG import salvaPNG


def _to_numpy(array):
    if type(array).__module__.split(".")[0] == "cupy":
        import cupy as cp

        return cp.asnumpy(array)
    return np.asarray(array)


def _matlab_round_nonnegative(values):
    """Equivale a ``round`` de MATLAB para las coordenadas no negativas."""
    return np.floor(values + 0.5)


def _read_histograms(base_dir, dataset, number_images, grid_size,
                     save_figures, output_dir):
    if dataset not in (1, 2):
        raise ValueError("dataset debe ser 1 (Peyre) o 2 (Altschuler).")

    import matplotlib.pyplot as plt

    Q = np.zeros((grid_size * grid_size, number_images), dtype=float)
    fig = None
    axes = None
    if save_figures:
        fig, axes = plt.subplots(5, 5, figsize=(10, 10))
        axes = axes.ravel()

    for index in range(number_images):
        number = index + 1
        if dataset == 1:
            path = base_dir / "dataPeyre" / f"{number}.csv"
            data = np.loadtxt(path, delimiter=",")
            data = np.atleast_2d(data).astype(float)
            total = data[:, 2].sum()
            if total <= 0:
                raise ValueError(f"Las masas de {path.name} no suman positivo.")
            data[:, 2] /= total
        else:
            path = base_dir / "dataAltschuler" / f"{number}.txt"
            data = np.loadtxt(path)
            data = np.atleast_2d(data).astype(float)
            data[:, :2] = _matlab_round_nonnegative(
                grid_size * data[:, :2]
            )

        image = np.zeros((grid_size, grid_size), dtype=float)
        for row in data:
            matlab_i = int(row[0])
            matlab_j = max(int(row[1]), 1)
            if not (1 <= matlab_i <= grid_size and 1 <= matlab_j <= grid_size):
                raise IndexError(
                    f"Coordenada ({matlab_i}, {matlab_j}) fuera de la "
                    f"grilla en {path}."
                )
            image[matlab_i - 1, matlab_j - 1] = row[2]

        Q[:, index] = image.reshape(-1, order="F")

        if save_figures and index < 25:
            axes[index].imshow(1.0 - image, cmap="hot", origin="upper")
            axes[index].set_xticks([])
            axes[index].set_yticks([])
            axes[index].set_title(str(number))

    if save_figures:
        for axis in axes[min(number_images, 25):]:
            axis.axis("off")
        fig.tight_layout()
        salvaPNG(fig, output_dir / "input-histograms.png")
        plt.close(fig)

    return Q


def main(
    methods=(3,),
    dataset=1,
    exact_wb=False,
    use_gpu=False,
    max_cpu=30.0,
    print_every=5.0,
    tol=-np.inf,
    number_images=10,
    grid_size=60,
    rho=5e3,
    lambda_=300.0,
    base_dir=None,
    output_dir=None,
    save_figures=True,
):
    """Ejecuta uno o más métodos del programa original.

    Métodos: 1=IBP, 2=MAM, 3=MAM-R, 4=MAM-H y 5=LP.
    ``base_dir`` debe contener ``dataPeyre`` o ``dataAltschuler``.
    """
    if isinstance(methods, (int, np.integer)):
        methods = (int(methods),)
    methods = tuple(int(method) for method in methods)
    if any(method not in (1, 2, 3, 4, 5) for method in methods):
        raise ValueError("Los métodos posibles son 1, 2, 3, 4 y 5.")

    base_dir = (
        Path(base_dir) if base_dir is not None
        else Path(__file__).resolve().parent
    )
    output_dir = (
        Path(output_dir) if output_dir is not None else base_dir / "Fig"
    )
    output_dir.mkdir(parents=True, exist_ok=True)

    number_images = int(number_images)
    grid_size = int(grid_size)

    print("Computing distance...")
    if exact_wb:
        kn = number_images * (grid_size - 1) + 1
        R = kn * kn
        D = distGrid(grid_size, number_images) ** 2 / (grid_size**2)
    else:
        kn = grid_size
        R = grid_size * grid_size
        D = distGrid(grid_size, 1) ** 2 / (grid_size**2)

    print("Reading data...")
    Q = _read_histograms(
        base_dir,
        dataset,
        number_images,
        grid_size,
        save_figures,
        output_dir,
    )

    results = {}
    for method in methods:
        if method == 1:
            if exact_wb:
                raise ValueError(
                    "El método IBP requiere que D y Q tengan igual número "
                    "de filas; use exact_wb=False."
                )
            print("Running IBP!")
            median = np.median(D)
            if median <= 0:
                raise ValueError("La mediana de D debe ser positiva.")
            p, _, _ = bregmanWassersteinBarycenter(
                Q,
                D / median,
                max_cpu,
                lambda_,
                use_gpu,
                tol,
                save_figures=save_figures,
                output_dir=output_dir,
                PrintEvery=print_every,
            )
        else:
            print("Arranging data...")
            supports = []
            q = []
            distances = []
            p_initial = np.ones(R, dtype=float) / R

            for index in range(number_images):
                mask = Q[:, index] > 1e-15
                support_size = int(mask.sum())
                if support_size == 0:
                    raise ValueError(
                        f"El histograma {index + 1} tiene soporte vacío."
                    )
                supports.append(support_size)
                distances.append(D[:, mask])
                q_vector = Q[mask, index].copy()
                q.append(q_vector / q_vector.sum())

            supports = np.asarray(supports, dtype=int)
            common = dict(
                d=distances,
                q=q,
                M=number_images,
                R=R,
                S=supports,
                p=p_initial,
                rho=rho,
                UseGPU=use_gpu,
                tol=tol,
                MaxCPU=max_cpu,
                PrintEvery=print_every,
                get_plan=GetPlan,
                save_figures=save_figures,
                output_dir=output_dir,
            )

            if method == 2:
                print("Running MAM!")
                p, _, _, _ = MAM(**common)
            elif method == 3:
                print("Running MAM-R!")
                p, _, _, _ = MAMR(**common)
            elif method == 4:
                print("Running MAM-H!")
                p, _, _, _ = MAMH(**common)
            else:
                print("Running LP with SciPy/HiGHS!")
                p, _, _, _, _ = LP_WB(
                    distances,
                    q,
                    number_images,
                    R,
                    supports,
                )

        p_array = _to_numpy(p).reshape((kn, kn), order="F")
        results[method] = p_array

        if save_figures:
            import matplotlib.pyplot as plt

            fig, ax = plt.subplots()
            ax.imshow(1.0 - p_array, cmap="hot", origin="upper")
            ax.set_title(f"Final method {method}")
            salvaPNG(fig, output_dir / f"Final-met-{method}.png")
            plt.close(fig)

    return results


if __name__ == "__main__":
    import argparse

    parser = argparse.ArgumentParser(
        description="Baricentro de Wasserstein: conversión de main.m"
    )
    parser.add_argument("--method", type=int, nargs="+", default=[3])
    parser.add_argument("--dataset", type=int, choices=(1, 2), default=1)
    parser.add_argument("--exact-wb", action="store_true")
    parser.add_argument("--use-gpu", action="store_true")
    parser.add_argument("--max-cpu", type=float, default=30.0)
    parser.add_argument("--print-every", type=float, default=5.0)
    parser.add_argument("--tol", type=float, default=-np.inf)
    parser.add_argument("--images", type=int, default=10)
    parser.add_argument("--grid-size", type=int, default=60)
    parser.add_argument("--rho", type=float, default=5e3)
    parser.add_argument("--lambda-value", type=float, default=300.0)
    parser.add_argument("--base-dir", type=Path)
    parser.add_argument("--output-dir", type=Path)
    parser.add_argument("--no-figures", action="store_true")
    args = parser.parse_args()

    main(
        methods=args.method,
        dataset=args.dataset,
        exact_wb=args.exact_wb,
        use_gpu=args.use_gpu,
        max_cpu=args.max_cpu,
        print_every=args.print_every,
        tol=args.tol,
        number_images=args.images,
        grid_size=args.grid_size,
        rho=args.rho,
        lambda_=args.lambda_value,
        base_dir=args.base_dir,
        output_dir=args.output_dir,
        save_figures=not args.no_figures,
    )


ModuleNotFoundError: No module named 'bregmanWassersteinBarycenter'

In [16]:
"""Conversión a Python de ``GetPlan.m``."""

import numpy as np


def _array_module(array):
    """Detecta NumPy o CuPy sin exigir CuPy en equipos que usan CPU."""
    if type(array).__module__.split(".")[0] == "cupy":
        import cupy as cp

        return cp
    return np


def GetPlan(p, q, R=None, S=None):
    """Construye un plan con marginales ``p`` y ``q``.

    Es la traducción de

    ``pi = repmat(p, 1, S)/S + repmat(q', R, 1)/R - 1/(R*S)``.

    Si ``p`` y ``q`` son vectores de probabilidad, las sumas por fila de
    ``pi`` son ``p`` y las sumas por columna son ``q``.
    """
    xp = _array_module(p)
    p = xp.asarray(p, dtype=float).reshape(-1)
    q = xp.asarray(q, dtype=float).reshape(-1)

    if R is None:
        R = p.size
    if S is None:
        S = q.size
    R = int(R)
    S = int(S)

    if p.size != R:
        raise ValueError(f"p debe tener longitud R={R}; tiene {p.size}.")
    if q.size != S:
        raise ValueError(f"q debe tener longitud S={S}; tiene {q.size}.")

    return p[:, None] / S + q[None, :] / R - 1.0 / (R * S)


get_plan = GetPlan

__all__ = ["GetPlan", "get_plan"]
